# Phase 1 Final Comparator Runner Readiness

Notebook fix marker: `phase1_final_comparator_runner_readiness_v2_latest_sources_20260421`

Purpose: record the final comparator runner/output-manifest readiness contract after reviewed final split, final feature and manifest-level leakage artifacts exist.

Scientific integrity rule: this notebook does **not** run final comparators, generate logits, compute metrics, audit runtime comparator logs, promote smoke outputs, or open claims. It records missing final comparator outputs honestly so downstream governance remains blocked until final runners and runtime audits exist.


## Technical Basis And Boundary

This notebook follows the repository contract:

- `configs/phase1/final_comparator_runner_readiness.json`: required final comparator runner/output-manifest boundary.
- `configs/phase1/final_comparator_artifacts.json`: required comparator artifact schema from the final claim-package plan.
- `final_split_manifest.json`: reviewed LOSO fold definitions.
- `final_feature_manifest.json`: reviewed feature schema/provenance only; it must not contain feature values, model outputs or metrics.
- `final_leakage_audit.json`: manifest-level leakage audit only; runtime comparator logs are still missing until final runners execute.
- `src/phase1/final_comparator_runner_readiness.py`: CLI runner that records the output-manifest contract and missing final comparator outputs.

Allowed interpretation: final comparator runners now have an audited input/output manifest contract. Not allowed: treating this readiness artifact as decoder efficacy, A3/A4 evidence, privileged-transfer evidence, or a final comparator performance result.


In [ ]:
# Cell 1 - Runtime setup, Drive mount, clone/pull repo, and unit tests.

from pathlib import Path
import base64
import getpass
import json
import subprocess

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752') if IN_COLAB else Path.cwd()
DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752') if IN_COLAB else Path.cwd()

def run(cmd, cwd=None, check=True):
    display = []
    for item in map(str, cmd):
        display.append('<redacted>' if 'Authorization: Basic' in item else item)
    print('$ ' + ' '.join(display))
    result = subprocess.run(list(map(str, cmd)), cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.stdout:
        print(result.stdout)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

if IN_COLAB and not REPO_DIR.exists():
    token = getpass.getpass('Nhap GitHub token cho repo private: ')
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    run(['git', '-c', f'http.extraHeader=Authorization: Basic {auth}', 'clone', REPO_URL, REPO_DIR])
elif REPO_DIR.exists():
    print(REPO_DIR)

run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)
run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)
unit_result = run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR, check=False)
if unit_result.returncode != 0:
    print('Unit tests failed. Do not generate or review final comparator runner readiness artifacts until tests pass.')
    raise subprocess.CalledProcessError(unit_result.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])


In [ ]:
# Cell 2 - Fixed paths and expected locked prereg identity.
# Update these reviewed run paths only when newer runs have been intentionally reviewed.

EXPECTED_PREREG_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
PREREG_BUNDLE = DRIVE_ROOT / 'artifacts/prereg/20260418T161442014597Z/prereg_bundle.json'
def latest_run(root, fallback):
    latest = root / 'latest.txt'
    return Path(latest.read_text(encoding='utf-8').strip()) if latest.exists() else fallback

FINAL_SPLIT_RUN = latest_run(DRIVE_ROOT / 'artifacts/phase1_final_split_manifest', DRIVE_ROOT / 'artifacts/phase1_final_split_manifest/20260421T082928310517Z')
FINAL_FEATURE_RUN = latest_run(DRIVE_ROOT / 'artifacts/phase1_final_feature_manifest', DRIVE_ROOT / 'artifacts/phase1_final_feature_manifest/20260421T084604606845Z')
FINAL_LEAKAGE_RUN = latest_run(DRIVE_ROOT / 'artifacts/phase1_final_leakage_audit', DRIVE_ROOT / 'artifacts/phase1_final_leakage_audit/20260421T085953617562Z')
OUTPUT_ROOT = DRIVE_ROOT / 'artifacts/phase1_final_comparator_runner_readiness'

print('Prereg bundle:', PREREG_BUNDLE)
print('Final split run:', FINAL_SPLIT_RUN)
print('Final feature run:', FINAL_FEATURE_RUN)
print('Final leakage run:', FINAL_LEAKAGE_RUN)
print('Output root:', OUTPUT_ROOT)


In [ ]:
# Cell 3 - Validate prereg, final split, final feature, and manifest-level leakage sources.

import hashlib


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

assert PREREG_BUNDLE.exists(), f'Missing prereg bundle: {PREREG_BUNDLE}'
bundle = read_json(PREREG_BUNDLE)
actual_prereg_hash = bundle.get('prereg_bundle_hash_sha256')
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', actual_prereg_hash)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert actual_prereg_hash == EXPECTED_PREREG_HASH, 'Prereg identity hash mismatch.'

split_summary = read_json(FINAL_SPLIT_RUN / 'phase1_final_split_manifest_summary.json')
split_validation = read_json(FINAL_SPLIT_RUN / 'phase1_final_split_manifest_validation.json')
split_manifest = read_json(FINAL_SPLIT_RUN / 'final_split_manifest.json')
feature_summary = read_json(FINAL_FEATURE_RUN / 'phase1_final_feature_manifest_summary.json')
feature_validation = read_json(FINAL_FEATURE_RUN / 'phase1_final_feature_manifest_validation.json')
feature_manifest = read_json(FINAL_FEATURE_RUN / 'final_feature_manifest.json')
leakage_summary = read_json(FINAL_LEAKAGE_RUN / 'phase1_final_leakage_audit_summary.json')
leakage_validation = read_json(FINAL_LEAKAGE_RUN / 'phase1_final_leakage_audit_validation.json')
leakage_audit = read_json(FINAL_LEAKAGE_RUN / 'final_leakage_audit.json')

print('Split status:', split_summary.get('status'), split_summary.get('split_manifest_ready'))
print('Feature status:', feature_summary.get('status'), feature_summary.get('feature_manifest_ready'))
print('Leakage status:', leakage_summary.get('status'), leakage_summary.get('leakage_audit_ready'))
print('Feature boundary:', {
    'contains_feature_matrix': feature_manifest.get('contains_feature_matrix'),
    'contains_model_outputs': feature_manifest.get('contains_model_outputs'),
    'contains_metrics': feature_manifest.get('contains_metrics'),
})
print('Leakage boundary:', {
    'outer_test_subject_used_in_any_fit': leakage_audit.get('outer_test_subject_used_in_any_fit'),
    'test_time_privileged_or_teacher_outputs_allowed': leakage_audit.get('test_time_privileged_or_teacher_outputs_allowed'),
    'runtime_comparator_logs_audited': leakage_audit.get('runtime_comparator_logs_audited'),
})

assert split_summary.get('status') == 'phase1_final_split_manifest_recorded'
assert split_summary.get('split_manifest_ready') is True
assert split_validation.get('status') == 'phase1_final_split_manifest_validation_passed'
assert split_validation.get('no_subject_overlap_between_train_and_test') is True
assert split_manifest.get('claim_ready') is False
assert split_manifest.get('smoke_artifacts_promoted') is False
assert feature_summary.get('status') == 'phase1_final_feature_manifest_recorded'
assert feature_summary.get('feature_manifest_ready') is True
assert feature_validation.get('status') == 'phase1_final_feature_manifest_validation_passed'
assert feature_manifest.get('contains_feature_matrix') is False
assert feature_manifest.get('contains_model_outputs') is False
assert feature_manifest.get('contains_metrics') is False
assert feature_manifest.get('claim_ready') is False
assert leakage_summary.get('status') == 'phase1_final_leakage_audit_recorded'
assert leakage_summary.get('leakage_audit_ready') is True
assert leakage_validation.get('status') == 'phase1_final_leakage_audit_validation_passed'
assert leakage_audit.get('outer_test_subject_used_in_any_fit') is False
assert leakage_audit.get('test_time_privileged_or_teacher_outputs_allowed') is False
assert leakage_audit.get('runtime_comparator_logs_audited') is False
assert leakage_audit.get('contains_model_outputs') is False
assert leakage_audit.get('contains_metrics') is False
assert leakage_audit.get('claim_ready') is False


In [ ]:
# Cell 4 - Source hash inventory for reproducibility.

HASH_TARGETS = [
    'configs/phase1/final_comparator_runner_readiness.json',
    'configs/phase1/final_comparator_artifacts.json',
    'src/phase1/final_comparator_runner_readiness.py',
    'src/phase1/final_split_manifest.py',
    'src/phase1/final_feature_manifest.py',
    'src/phase1/final_leakage_audit.py',
    'src/cli.py',
    'tests/unit/test_phase1_final_comparator_runner_readiness.py',
    'notebooks/22_colab_phase1_final_comparator_runner_readiness.ipynb',
]
source_hashes = {}
for rel in HASH_TARGETS:
    path = REPO_DIR / rel
    assert path.exists(), f'Missing source target: {rel}'
    source_hashes[rel] = hashlib.sha256(path.read_bytes()).hexdigest()
print(json.dumps(source_hashes, indent=2))


In [ ]:
# Cell 5 - Run the CLI-backed final comparator runner readiness package.

cmd = [
    'python', '-m', 'src.cli', 'phase1_final_comparator_runner_readiness',
    '--profile', 't4_safe',
    '--config', str(PREREG_BUNDLE),
    '--final-split-run', str(FINAL_SPLIT_RUN),
    '--final-feature-run', str(FINAL_FEATURE_RUN),
    '--final-leakage-run', str(FINAL_LEAKAGE_RUN),
    '--output-root', str(OUTPUT_ROOT),
    '--runner-config', 'configs/phase1/final_comparator_runner_readiness.json',
    '--artifact-config', 'configs/phase1/final_comparator_artifacts.json',
]
run(cmd, cwd=REPO_DIR)
print('Final comparator runner readiness command completed. Review artifacts before implementing final runners.')


In [ ]:
# Cell 6 - Inspect latest output and verify required artifacts.

latest = OUTPUT_ROOT / 'latest.txt'
print('latest exists:', latest.exists())
assert latest.exists(), 'No latest.txt written for final comparator runner readiness output.'
run_dir = Path(latest.read_text(encoding='utf-8').strip())
print('Latest final comparator runner readiness output:', run_dir)
assert run_dir.exists(), f'Output run directory does not exist: {run_dir}'

required = [
    'phase1_final_comparator_runner_readiness_inputs.json',
    'phase1_final_comparator_runner_readiness_summary.json',
    'phase1_final_comparator_runner_readiness_report.md',
    'phase1_final_comparator_runner_source_links.json',
    'phase1_final_comparator_runner_input_validation.json',
    'phase1_final_comparator_runner_output_contract.json',
    'phase1_final_comparator_runner_manifest_status.json',
    'phase1_final_comparator_missing_outputs.json',
    'phase1_final_comparator_runtime_leakage_requirements.json',
    'phase1_final_comparator_completeness_table.json',
    'phase1_final_comparator_runner_claim_state.json',
    'phase1_final_comparator_runner_implementation_plan.json',
]
for name in required:
    path = run_dir / name
    print(name, 'exists =', path.exists())
    assert path.exists(), f'Missing required artifact: {name}'

summary = read_json(run_dir / 'phase1_final_comparator_runner_readiness_summary.json')
input_validation = read_json(run_dir / 'phase1_final_comparator_runner_input_validation.json')
contract = read_json(run_dir / 'phase1_final_comparator_runner_output_contract.json')
manifest_status = read_json(run_dir / 'phase1_final_comparator_runner_manifest_status.json')
missing_outputs = read_json(run_dir / 'phase1_final_comparator_missing_outputs.json')
runtime_requirements = read_json(run_dir / 'phase1_final_comparator_runtime_leakage_requirements.json')
completeness = read_json(run_dir / 'phase1_final_comparator_completeness_table.json')
claim_state = read_json(run_dir / 'phase1_final_comparator_runner_claim_state.json')
print(json.dumps({
    'run_dir': str(run_dir),
    'summary_status': summary.get('status'),
    'upstream_manifests_ready': summary.get('upstream_manifests_ready'),
    'final_comparator_outputs_present': summary.get('final_comparator_outputs_present'),
    'runtime_comparator_logs_audited': summary.get('runtime_comparator_logs_audited'),
    'claim_ready': summary.get('claim_ready'),
    'headline_phase1_claim_open': summary.get('headline_phase1_claim_open'),
    'required_final_comparators': summary.get('required_final_comparators'),
    'manifest_status': summary.get('manifest_status'),
    'claim_blockers': summary.get('claim_blockers'),
}, indent=2))


In [ ]:
# Cell 7 - Comparator output-manifest boundary assertions.

assert summary.get('status') == 'phase1_final_comparator_runner_readiness_recorded'
assert summary.get('upstream_manifests_ready') is True
assert summary.get('claim_ready') is False
assert summary.get('headline_phase1_claim_open') is False
assert summary.get('full_phase1_claim_bearing_run_allowed') is False
assert summary.get('final_comparator_outputs_present') is False
assert summary.get('all_comparator_output_manifests_present') is False
assert summary.get('runtime_comparator_logs_audited') is False
assert summary.get('smoke_artifacts_promoted') is False
assert input_validation.get('status') == 'phase1_final_comparator_runner_inputs_ready'
assert contract.get('comparator_contract_matches_artifact_plan') is True
assert contract.get('output_schema_matches_artifact_plan') is True
assert manifest_status.get('status') == 'phase1_final_comparator_outputs_missing'
assert manifest_status.get('claim_evaluable') is False
assert completeness.get('claim_evaluable') is False
assert completeness.get('all_required_outputs_present') is False
assert runtime_requirements.get('claim_evaluable') is False
assert missing_outputs.get('claim_evaluable') is False

expected_comparators = {'A2', 'A2b', 'A2c_CORAL', 'A2d_riemannian', 'A3_distillation', 'A4_privileged'}
observed_comparators = {row['comparator_id'] for row in manifest_status['comparators']}
assert observed_comparators == expected_comparators
for row in manifest_status['comparators']:
    assert row['claim_evaluable'] is False
    assert row['final_fold_logs'] == 'missing'
    assert row['final_logits'] == 'missing'
    assert row['final_subject_level_metrics'] == 'missing'
    assert row['runtime_leakage_logs'] == 'missing'
    assert row['comparator_output_manifest'] == 'missing'
    assert row['teacher_or_privileged_test_time_outputs_allowed'] is False
    assert row['smoke_metrics_promoted'] is False
    assert 'runtime_leakage_logs' in row['missing_outputs']

for row in completeness['rows']:
    assert row['claim_evaluable'] is False
    assert row['final_fold_logs_present'] is False
    assert row['final_logits_present'] is False
    assert row['final_subject_level_metrics_present'] is False
    assert row['runtime_leakage_logs_present'] is False
    assert row['comparator_output_manifest_present'] is False

assert 'final_comparator_outputs_missing' in claim_state['blockers']
assert 'runtime_comparator_leakage_logs_missing_until_final_runners_execute' in claim_state['blockers']
print('Final comparator runner readiness review passed: output manifests are explicitly missing and claims remain closed.')


In [ ]:
# Cell 8 - Write a non-claim review note.

from datetime import datetime, timezone

review_note = {
    'status': 'phase1_final_comparator_runner_readiness_review_pass_non_claim',
    'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
    'reviewed_run': str(run_dir),
    'scope': 'Phase 1 final comparator runner/output-manifest readiness only',
    'checks_passed': [
        'required_artifacts_present',
        'upstream_final_split_manifest_ready',
        'upstream_final_feature_manifest_ready',
        'upstream_manifest_level_leakage_audit_ready',
        'contract_matches_final_comparator_artifact_plan',
        'final_comparator_outputs_marked_missing',
        'runtime_comparator_logs_marked_missing',
        'all_comparator_output_manifests_claim_evaluable_false',
        'smoke_metrics_promoted_false',
        'claim_ready_false',
        'headline_phase1_claim_open_false',
    ],
    'required_final_comparators': summary.get('required_final_comparators'),
    'manifest_status': manifest_status.get('status'),
    'final_comparator_outputs_present': summary.get('final_comparator_outputs_present'),
    'runtime_comparator_logs_audited': summary.get('runtime_comparator_logs_audited'),
    'claim_blockers': summary.get('claim_blockers'),
    'allowed_interpretation': claim_state.get('allowed_interpretation'),
    'not_ok_to_claim': claim_state.get('not_ok_to_claim'),
    'next_allowed_steps': [
        'Implement final feature matrix materialization and final comparator runners against this contract.',
        'Require each final comparator to write fold logs, logits, subject-level metrics, runtime leakage logs and output manifest.',
        'Do not feed controls/calibration/influence/reporting from smoke diagnostics.',
        'Do not open headline claims until final comparator outputs, runtime leakage logs, controls, calibration, influence and reporting all pass locked rules.',
    ],
}
review_path = run_dir / 'phase1_final_comparator_runner_readiness_review_note.json'
review_path.write_text(json.dumps(review_note, indent=2), encoding='utf-8')
print('Review note written:', review_path)
print(json.dumps(review_note, indent=2))


In [ ]:
# Cell 9 - Closeout.

print('================ Phase 1 Final Comparator Runner Readiness Closeout ================')
print('Notebook fix marker: phase1_final_comparator_runner_readiness_v2_latest_sources_20260421')
print('Latest runner readiness output:', run_dir)
print('Upstream manifests ready:', summary.get('upstream_manifests_ready'))
print('Required final comparators:', summary.get('required_final_comparators'))
print('Final comparator outputs present:', summary.get('final_comparator_outputs_present'))
print('All comparator output manifests present:', summary.get('all_comparator_output_manifests_present'))
print('Runtime comparator logs audited:', summary.get('runtime_comparator_logs_audited'))
print('Smoke artifacts promoted:', summary.get('smoke_artifacts_promoted'))
print('Claim blockers:', summary.get('claim_blockers'))
print('')
print('CHECK REQUIRED: final comparator output manifests are intentionally marked missing. Implement final runners next; do not treat this readiness package as model evidence.')
print('')
print('NOT OK TO CLAIM: This readiness notebook does not prove decoder efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy, or full Phase 1 neural comparator performance.')
